# Pre-body

## Clearing past runs (optional)

In [1]:
# !rm -rf logs/ # clear logs
# !rm -rf optimizer_output/

## IIC-OSIC Env Setup

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [3]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-10-01 22:15:41,651 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-10-01 22:15:41,657 - SymXplorer.jupyter - Spicelib_Wrapper imported successfully.


# Instantiations


## Loading the project config

In [4]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

22:15:41 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
22:15:41 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-01_22-15-41.log
22:15:41 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
22:15:41 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
22:15:41 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=10, random_seed=48
22:15:41 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
22:15:41 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
22:15:41 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
22:15:41 - SymXplorer.domains: [INFO] 	Number of target specs: 3
22:15:41 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=100e6, tolerance=100000.0, goal=exact, 

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.01), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.01), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06)}), pvt=PVT(temp=25, corner='tt', supply=

## Create a SPICE simulator wrapper

In [5]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

22:15:41 - SymXplorer.spicelib: [WARNING] ⚠️ Output directory already exists, re-creating: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
22:15:41 - SymXplorer.spicelib: [INFO] --------------------------------------------------
22:15:41 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
22:15:41 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
22:15:41 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
22:15:41 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
22:15:41 - SymXplorer.spicelib: [INFO] --------------------------------------------------
22:15:41 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
22:15:41 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
22:15:41 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
22:15:41 - SymXplorer.spicelib: [INFO] Te

## Create an optimizer object

In [6]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

22:15:41 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs


## Sanity Check

In [7]:
# wrapper.run_sanity_check(
#     use_editor=True,
#     sim_execution_t=Sim_Execution_Type.RUN_NOW
# )

# Main Body

## Optimization

In [8]:
circuit_optimizer.parameterize()

Dict(vbias=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 50.0, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0, 'vbias': 50.0}

In [9]:
circuit_optimizer.optimize()

22:15:42 - SymXplorer.optimizer: [INFO] Optimization process started.
22:15:42 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 10
Optimizing: 100%|██████████| 10/10 [00:05<00:00,  1.88trial/s]
22:15:47 - SymXplorer.optimizer: [INFO] Optimization process completed.


[{'params': {'x_dut_nfet_w': 44.11534062833345,
   'x_dut_nfet_l': 36.118785072336856,
   'x_dut_cap_w': 49.42175513146307,
   'x_dut_cap_l': 48.36794120698788,
   'x_dut_res_s_l': 41.61402179440088,
   'x_dut_res_s_w': 39.66694196306865,
   'x_dut_res_3_l': 53.60326510443906,
   'x_dut_res_3_w': 49.94026536861408,
   'vbias': 57.8164143627324},
  'loss': np.float64(1000000000915.8539),
  'metadata': {'fc': {'curr_val': np.float64(8414611.0),
    'loss': np.float64(915.85389)},
   'q': {'curr_val': np.float64(9.567254713355664), 'loss': np.float64(0.0)},
   'gain_db': {'curr_val': np.float64(-19.155154376881573),
    'loss': np.float64(1000000000000.0)}}},
 {'params': {'x_dut_nfet_w': 47.38948226847224,
   'x_dut_nfet_l': 51.866335362288446,
   'x_dut_cap_w': 67.76628482285233,
   'x_dut_cap_l': 56.045837138451354,
   'x_dut_res_s_l': 46.808877883658056,
   'x_dut_res_s_w': 58.07035513044152,
   'x_dut_res_3_l': 33.886163749470974,
   'x_dut_res_3_w': 60.89859643284487,
   'vbias': 47.

In [10]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

22:15:47 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
22:15:47 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [11]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

22:15:48 - SymXplorer.optimizer: [INFO] best loss: 1000000000882.6589


{'fc': {'curr_val': np.float64(11734110.0), 'loss': np.float64(882.6589)},
 'q': {'curr_val': np.float64(13.29192342546443),
  'loss': np.float64(1.6459617127322153)},
 'gain_db': {'curr_val': np.float64(-17.53225886684781),
  'loss': np.float64(1000000000000.0)}}

In [12]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param]*1e6 :0.2f}")

x_dut_nfet_w: 4.91
x_dut_nfet_l: 5.70
x_dut_cap_w: 3225.26
x_dut_cap_l: 3804.78
x_dut_res_s_l: 455.54
x_dut_res_s_w: 549.94
x_dut_res_3_l: 481.00
x_dut_res_3_w: 422.46
vbias: 796377.40


In [13]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

22:15:48 - SymXplorer.optimizer: [INFO] total loss: 1000000000882.6589
22:15:48 - SymXplorer.optimizer: [INFO] 	Spec 'fc': curr_val=11734110.0, loss=882.6589
22:15:48 - SymXplorer.optimizer: [INFO] 	Spec 'q': curr_val=13.29192342546443, loss=1.6459617127322153
22:15:48 - SymXplorer.optimizer: [INFO] 	Spec 'gain_db': curr_val=-17.53225886684781, loss=1000000000000.0


### (3) Metric Trace

In [14]:
circuit_optimizer.plot_optimization_trace(metric_x='fc', metric_y='gain_db', show=True)

22:15:49 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


(tensor([ 8414611.0000,  6682188.5000,  9140536.0000,  6397924.0000,
          6988935.0000,  9830020.5000,  8673445.5000, 10565795.0000,
         11734110.0000, 10103790.5000]),
 tensor([-19.1552, -26.5932, -23.0921, -30.4580, -25.5344, -19.6426, -19.1111,
         -16.4730, -17.5323, -20.3922]))

In [15]:
circuit_optimizer.plot_loss_value_by_spec(spec_name="gain_db", show=True)
circuit_optimizer.plot_loss_value_by_spec(spec_name="fc", show=True)

22:15:49 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


22:15:49 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


### (4) Design Space Exploration

In [16]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_w", param_y="x_dut_nfet_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_cap_l", param_y="x_dut_cap_w", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="vbias", param_y="x_dut_cap_w", show=True)

22:15:49 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


22:15:49 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


22:15:49 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


(tensor([1.0407, 0.8541, 0.6135, 0.7867, 0.8410, 0.9014, 0.8264, 0.9606, 0.7964,
         0.7715]),
 tensor([0.0049, 0.0068, 0.0045, 0.0064, 0.0062, 0.0036, 0.0044, 0.0039, 0.0032,
         0.0035]))

# Testing

## Other

In [17]:
circuit_optimizer.optimization_log

[{'metric_value': np.float64(1000000000915.8539),
  'fit_summary': {'fc': {'curr_val': np.float64(8414611.0),
    'loss': np.float64(915.85389)},
   'q': {'curr_val': np.float64(9.567254713355664), 'loss': np.float64(0.0)},
   'gain_db': {'curr_val': np.float64(-19.155154376881573),
    'loss': np.float64(1000000000000.0)}},
  'params': {'x_dut_nfet_w': 4.5121264497023446e-06,
   'x_dut_nfet_l': 3.726864694103479e-06,
   'x_dut_cap_w': 0.004942681295594994,
   'x_dut_cap_l': 0.004837310441286719,
   'x_dut_res_s_l': 0.00041672407772606483,
   'x_dut_res_s_w': 0.00039727275021105593,
   'x_dut_res_3_l': 0.0005364966183933463,
   'x_dut_res_3_w': 0.0004999032510324547,
   'vbias': 1.0406954585291832},
  'log': PosixPath('/foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/run_1/tb_ac_1.log')},
 {'metric_value': np.float64(1000000000933.1781),
  'fit_summary': {'fc': {'curr_val': np.float64(6682188.5),
    'loss': np.float64(933.178115)},
   'q': {'curr_val': np.flo

In [18]:
PROJECT_SETUP.optimizer_config.target_specs.list_target_names()

['fc', 'q', 'gain_db']

In [19]:
target_spec = PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name('gain_db')
target_spec

TargetSpec(name='gain_db', target=40, goal=<OptimizationGoalType.EXCEED: 'exceed'>, sim_type=<SimType.AC: 'ac'>, log_scale=False, enable=True, error_type=<Error_Types.RELATIVE_EXPONENTIAL: 'relative-exponential'>, weight=1.0, tolerance=1, description='gain in dB at fc')

In [20]:
circuit_optimizer.compute_spec_loss(curr_val=-90, target_spec=target_spec)

np.float64(2.872649550817832e+56)

In [21]:
PROJECT_SETUP.dut_params

[Param(name='x_dut_nfet_w', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False),
 Param(name='x_dut_nfet_l', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False),
 Param(name='x_dut_cap_w', min_val=np.float64(1e-06), max_val=np.float64(0.01), val=None, description=None, log_scale=False),
 Param(name='x_dut_cap_l', min_val=np.float64(1e-06), max_val=np.float64(0.01), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_s_l', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_s_w', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_3_l', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_3_w', min_val=np.float64(1e-06), max_val=np.fl